# Práctica 4 · Detección de anomalías: ¿qué incidentes se comportan de forma rara?
**Analítica de procesos para ciberseguridad**

**El problema.** Casi todo ataque, fallo o abuso **se sale de lo normal** de alguna manera: un incidente que rebota entre equipos 25 veces, uno que se reabre una y otra vez, uno que nadie toca en tres meses. Si sabemos cómo es lo **normal**, lo **raro** se vuelve medible. Un analista no puede revisar miles de incidentes: ¿podemos señalarle los **pocos** que más se desvían?

**La tarea.** Igual que en la Práctica 3, **no hay etiqueta** que nos diga «este es anómalo». Un modelo aprende cómo se ve **lo común** y puntúa qué tanto **se aleja** cada incidente. Es **detección de anomalías**.

**Una analogía.** Un guardia de edificio que lleva años en su puesto conoce el ritmo normal: quién entra, a qué hora, con qué cosas. No sabe qué es un ladrón, pero **nota cuando algo no encaja**. Eso mismo es un detector de anomalías: no sabe qué es un ataque, **solo sabe qué es lo habitual**.

> **Muy importante:** **anómalo no significa malicioso.** Un incidente raro puede ser un ataque, un error, un caso excepcional legítimo o un dato mal registrado. El detector **señala dónde mirar**; la decisión es de una persona.

**Los datos.** El mismo registro de incidentes de las Prácticas 2 y 3 (mesa de servicio de TI, ServiceNow; 24.916 incidentes), resumido en una fila por incidente.

> **Fuente y cambios.** Amaral, Fantinato, Reijers y Peres (2018). *Incident management process enriched event log.* UCI Machine Learning Repository, https://doi.org/10.24432/C57S4H. Licencia CC BY 4.0. Cambios hechos con fines académicos (traducción al español, formato de fechas, 2 incidentes inválidos fuera): ver `scripts/preparar_servicenow.py`.

**Qué vas a hacer**
1. Probar la regla más simple y ver por qué se queda corta.
2. Probar dos modelos: **Isolation Forest** y un **autoencoder** (una red neuronal).
3. Resolver el problema difícil: **¿cómo saber si funcionan, si no hay respuesta correcta?**
4. Ver qué es «raro **para su grupo**».

## 0 · Preparación

In [ ]:
# Importa las librerías: pandas (tablas), numpy (cálculos) y matplotlib (gráficos)
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")                # oculta advertencias para que la salida se lea limpia
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# Dirección de los datos: repositorio de GitHub y versión fija de donde se descargan los archivos
REPO = "lsandinop/analitica-procesos-cybersecurity"
VERSION = "main"
DATA_BASE = "" if REPO.startswith("USUARIO") else f"https://raw.githubusercontent.com/{REPO}/{VERSION}/datos/"

def cargar(nombre, fechas=None):
    # Lee un archivo CSV y lo devuelve como tabla (DataFrame). Lo busca primero en GitHub;
    # si no está disponible, en la carpeta local. `fechas` son las columnas que se leen como fecha y hora.
    if DATA_BASE:
        try:
            return pd.read_csv(DATA_BASE + nombre, parse_dates=fechas)
        except Exception:
            print("No pude descargar el archivo de GitHub; busco una copia local.")
    for ruta in (nombre, "datos/" + nombre, "../datos/" + nombre, "../../datos/" + nombre):
        if os.path.exists(ruta):
            return pd.read_csv(ruta, parse_dates=fechas)
    raise FileNotFoundError(f"No encuentro {nombre}: súbelo a la carpeta del cuaderno")

eventos = cargar("incidentes_servicenow.csv", fechas=["fecha_evento", "fecha_apertura", "fecha_resolucion", "fecha_cierre"])

def tabla_por_incidente(eventos):
    # Resume el registro de eventos en UNA fila por incidente, con variables que describen CÓMO se atendió (igual que en la Práctica 3)
    ev = eventos.sort_values(["incidente", "fecha_evento"], kind="stable")
    g = ev.groupby("incidente")
    cambios = ev[ev.estado != g.estado.shift()]                     # solo las filas donde cambia el estado
    t = pd.DataFrame({
        "cambios_de_estado": cambios.groupby("incidente").size(),
        "reasignaciones": g.reasignaciones.max(),
        "reaperturas": g.reaperturas.max(),
        "modificaciones": g.modificaciones.max(),
        "esperas": cambios.estado.str.startswith("Esperando").groupby(cambios.incidente).sum(),
        "grupos_distintos": g.grupo_asignado.nunique(),
        "dias_a_resolver": (g.fecha_resolucion.last() - g.fecha_apertura.first()).dt.total_seconds() / 86400,
        "prioridad": g.prioridad.last().str[0].astype(float),          # «3 - Moderada» -> 3
        "cumplio_sla": g.cumplio_sla.last()})
    return t.dropna()

tabla = tabla_por_incidente(eventos)
VARIABLES = ["cambios_de_estado", "reasignaciones", "reaperturas", "modificaciones", "esperas", "grupos_distintos", "dias_a_resolver", "prioridad"]
print(f"{len(tabla):,} incidentes con datos completos".replace(",", "."))
tabla.head(5)

## 1 · ¿Qué es «raro»? Tres tipos de anomalía
| Tipo | Qué es | Ejemplo en un SOC |
|---|---|---|
| **Puntual** | Un valor extremo | Un incidente abierto 300 días |
| **Contextual** | Normal en general, raro **en su contexto** | Un incidente de «rutina» que de pronto tarda 3 días |
| **Colectiva** | Un **conjunto** de eventos normales que, juntos, son raros | Cien inicios de sesión fallidos seguidos desde una misma dirección |

En esta práctica buscamos las dos primeras.

## 2 · Lo más simple: una regla estadística
La regla clásica: *«lo normal está cerca del promedio; lo que se aleja más de 3 desviaciones estándar es raro»*. Se supone que los datos forman una **campana** (distribución normal). Probémosla con los **días a resolver**:

In [ ]:
d = tabla.dias_a_resolver
media, desviacion = d.mean(), d.std()
limite = media + 3 * desviacion
raros_regla = d > limite
print(f"Media {media:.1f} días · desviación {desviacion:.1f} · límite (media + 3 desviaciones) = {limite:.0f} días")
print(f"La regla marca {int(raros_regla.sum())} incidentes ({raros_regla.mean():.1%})")

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(d.clip(upper=120), bins=80, color="#1f6db0")
ax.axvline(limite, color="#d9534f", ls="--"); ax.text(limite + 2, ax.get_ylim()[1] * 0.8, f"límite: {limite:.0f} días", color="#d9534f")
ax.set_xlabel("días a resolver (los mayores de 120 se agrupan en el último valor)"); ax.set_ylabel("incidentes"); ax.set_title("¿Forman una campana los días a resolver?")
plt.tight_layout(); plt.show()

**Qué leer ahí.** **No es una campana:** casi todo está pegado a cero y hay una cola larguísima. Eso rompe la regla de dos maneras:
1. Los casos extremos **inflan la desviación** (22 días, cuando la mitad de los incidentes se resuelve en menos de uno), así que el límite queda altísimo.
2. La regla mira **una sola variable**. Un incidente que rebotó entre 8 grupos 25 veces pero se resolvió en dos días **ni se entera**.

Veremos esto con números más abajo. Primero, dos modelos que miran **todas las variables a la vez**.

## 3 · Isolation Forest: aislar al raro
**La idea.** Imagina que quieres aislar a una persona en una multitud haciéndole preguntas al azar («¿tiene más de 40 años?», «¿lleva sombrero?»). A alguien común le harás **muchas** preguntas antes de quedarte con él solo; a alguien **raro**, **muy pocas**. *Isolation Forest* hace exactamente eso con cientos de árboles de preguntas al azar: **cuantas menos preguntas hagan falta para aislar a un incidente, más anómalo es**.

Antes, igual que en la segmentación, pasamos las variables a una escala común:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

def preparar(df, columnas):
    # Logaritmo (para domar los valores extremos) y estandarización (escala común)
    return StandardScaler().fit_transform(np.log1p(df[columnas]))

Z = preparar(tabla, VARIABLES)
bosque_aislante = IsolationForest(n_estimators=200, random_state=0).fit(Z)          # 200 árboles de preguntas al azar
puntaje_if = -bosque_aislante.score_samples(Z)                                      # más alto = más anómalo (se cambia el signo para que se lea así)
tabla["puntaje_isolation"] = puntaje_if
print("Los 10 incidentes más anómalos según Isolation Forest:")
top = tabla.sort_values("puntaje_isolation", ascending=False).head(10)[VARIABLES + ["puntaje_isolation"]]
top.round(2)

**Lee esa tabla como lo haría un analista.** Son incidentes con **muchas reasignaciones, muchos cambios de estado, reaperturas o muchos grupos**: se movieron más que cualquier otro. Eso es lo que significa «raro» **para este detector**. Y además son **largos** (más de tres semanas).

## 4 · Autoencoder: una red neuronal que aprende lo normal
**La idea.** Una red neuronal se entrena para **copiar su entrada**, pero pasando por un **cuello de botella**: primero comprime las 8 variables a solo 3 números y luego intenta reconstruir las 8. Como el cuello es estrecho, la red solo puede aprender **lo que es común** en los datos. Cuando le llega un incidente **raro**, no sabe reconstruirlo bien: **el error de reconstrucción es grande**. Ese error es el puntaje de anomalía.

| Término | Aquí significa |
|---|---|
| **Capas (6, 3, 6)** | Una capa de 6 neuronas que comprime, una de **3** (el cuello de botella) y otra de 6 que reconstruye |
| **Épocas** | Cuántas veces ve toda la tabla para aprender (máximo 200 aquí) |
| **Error de reconstrucción** | Qué tan distinta es la copia del original; si es grande, el incidente es raro |

In [ ]:
from sklearn.neural_network import MLPRegressor

autoencoder = MLPRegressor(hidden_layer_sizes=(6, 3, 6), max_iter=200, random_state=0).fit(Z, Z)     # entrada = salida: aprende a copiar
error = ((autoencoder.predict(Z) - Z) ** 2).mean(axis=1)                                             # cuánto se equivoca al copiar cada incidente
tabla["error_autoencoder"] = error
print("Los 10 incidentes más anómalos según el autoencoder:")
tabla.sort_values("error_autoencoder", ascending=False).head(10)[VARIABLES + ["error_autoencoder"]].round(2)

In [ ]:
# ¿Coinciden los dos métodos? Los 200 incidentes más anómalos de cada uno
top_if = set(tabla.sort_values("puntaje_isolation", ascending=False).head(200).index)
top_ae = set(tabla.sort_values("error_autoencoder", ascending=False).head(200).index)
print(f"De los 200 más anómalos de cada método, coinciden {len(top_if & top_ae)}")

**Qué leer ahí.** Los dos métodos coinciden en **menos de un tercio** de los incidentes. No es un error: cada uno tiene **su propia definición de «raro»**. *Isolation Forest* se fija en lo que es fácil de separar (incidentes que rebotaron entre muchos grupos); el autoencoder, en lo que no puede reconstruir (por ejemplo, incidentes con muchísimas modificaciones y cientos de días, pero sin reasignaciones). Por eso en ciberseguridad se suelen **combinar** varios detectores.

## 5 · El problema difícil: ¿cómo saber si funcionan?
En la Práctica 1 teníamos la respuesta del analista y era fácil calificar. Aquí **no hay respuesta correcta**. Hay tres salidas, ninguna perfecta:

### (a) Inyectar anomalías que conocemos
Fabricamos **200 incidentes anómalos** copiando incidentes reales y **alterándolos** (más reasignaciones, más esperas, más cambios de estado, más grupos) y los mezclamos con los reales. Luego preguntamos: de los 200 incidentes con **mayor puntaje**, ¿cuántos son los que fabricamos?

In [ ]:
rng = np.random.RandomState(7)
n_inyectados = 200
copias = tabla[VARIABLES + ["cumplio_sla"]].sample(n_inyectados, random_state=7).copy()           # incidentes reales que vamos a alterar
copias["reasignaciones"] += rng.randint(8, 20, n_inyectados); copias["esperas"] += rng.randint(3, 7, n_inyectados)
copias["cambios_de_estado"] += rng.randint(6, 12, n_inyectados); copias["grupos_distintos"] += rng.randint(3, 6, n_inyectados)
copias["modificaciones"] *= rng.randint(3, 6, n_inyectados); copias["dias_a_resolver"] *= rng.uniform(2, 5, n_inyectados)
copias.index = [f"SINTETICO{i}" for i in range(n_inyectados)]

mezcla = pd.concat([tabla[VARIABLES + ["cumplio_sla"]], copias])
es_inyectado = pd.Series(mezcla.index.astype(str)).str.startswith("SINTETICO").values
Zm = preparar(mezcla, VARIABLES)

puntajes = {"Regla: días (distancia a la media)": np.abs(StandardScaler().fit_transform(mezcla[["dias_a_resolver"]]))[:, 0],
            "Isolation Forest": -IsolationForest(n_estimators=200, random_state=0).fit(Zm).score_samples(Zm)}
ae_m = MLPRegressor(hidden_layer_sizes=(6, 3, 6), max_iter=200, random_state=0).fit(Zm, Zm)
puntajes["Autoencoder"] = ((ae_m.predict(Zm) - Zm) ** 2).mean(axis=1)

recall = {}
for nombre, p in puntajes.items():
    primeros = np.argsort(-p)[:n_inyectados]                                                       # los 200 con mayor puntaje
    recall[nombre] = es_inyectado[primeros].mean()
    print(f"{nombre:36s} atrapa {int(es_inyectado[primeros].sum())} de los {n_inyectados} inyectados ({recall[nombre]:.0%})")

ax = pd.Series(recall).iloc[::-1].mul(100).plot.barh(figsize=(8, 2.6), color=["#8e44ad", "#1f6db0", "#a8b0b6"]); ax.set_xlim(0, 100)
for i, v in enumerate(pd.Series(recall).iloc[::-1] * 100): ax.text(v + 1, i, f"{v:.0f} %", va="center")
plt.title("De los 200 inyectados, ¿cuántos quedan entre los 200 más anómalos?"); plt.xlabel("% atrapado"); plt.tight_layout(); plt.show()

**Qué leer ahí.** La regla sobre los días atrapa **casi ninguno**; *Isolation Forest* y el autoencoder, la gran mayoría. Mirar **varias variables a la vez** es lo que importa. **Pero cuidado:** este examen **solo mide lo que fabricamos**. Diseñamos anomalías con muchas variables extremas a la vez, y eso es justo lo que *Isolation Forest* detecta bien. No dice que atrape cualquier clase de rareza.

### (b) Un indicador indirecto
Sí tenemos un dato que **suele ir con lo raro**: el **incumplimiento del plazo (SLA)**. Si los incidentes que el detector marca incumplen mucho más que el promedio, es buena señal. Para no hacer trampa, **sacamos de las variables los días a resolver** (el plazo depende de ellos) y comprobamos:

In [ ]:
sin_dias = [v for v in VARIABLES if v != "dias_a_resolver"]
Zs = preparar(tabla, sin_dias)
p_if = -IsolationForest(n_estimators=200, random_state=0).fit(Zs).score_samples(Zs)
ae_s = MLPRegressor(hidden_layer_sizes=(6, 3, 6), max_iter=200, random_state=0).fit(Zs, Zs)
p_ae = ((ae_s.predict(Zs) - Zs) ** 2).mean(axis=1)
incumple = ~tabla.cumplio_sla.values

filas = []
for nombre, p in {"Isolation Forest": p_if, "Autoencoder": p_ae}.items():
    for k in (100, 500, 2000):
        filas.append({"método": nombre, "se revisan": k, "% que incumple el plazo": incumple[np.argsort(-p)[:k]].mean() * 100})
print(f"Base: {incumple.mean():.1%} de los incidentes incumple el plazo")
pd.DataFrame(filas).pivot(index="método", columns="se revisan", values="% que incumple el plazo").round(1)

**Qué leer ahí.** Entre los incidentes más anómalos, el incumplimiento del plazo es **mucho más alto que el promedio**, aunque el detector **nunca vio** ese dato ni los días. Es una señal de que **lo que marca tiene sentido**. Pero no es una prueba: el plazo no es lo mismo que «anómalo», y se puede incumplir por mil razones.

### (c) La revisión humana
La evaluación más honesta es la más cara: **un analista revisa los marcados y dice cuántos valía la pena mirar.** En una puesta en marcha real, se empieza así: se muestran los 20 primeros y se **mide la precisión de esa revisión** (¿cuántos de los 20 eran de verdad dignos de atención?).

## 6 · ¿Cuánto revisar?
El detector entrega un **puntaje**, no una decisión. Hay que escoger **un corte**: revisar el 1 %, el 2 % o el 5 % con mayor puntaje. Más revisión encuentra más cosas raras y cuesta más trabajo:

In [ ]:
total = len(mezcla)
p = puntajes["Isolation Forest"]
orden = np.argsort(-p)
filas = []
for pct in (0.5, 1, 2, 5, 10):
    k = int(total * pct / 100)
    filas.append({"se revisa el": f"{pct} %", "incidentes a revisar": k, "inyectados que se encuentran": int(es_inyectado[orden[:k]].sum()),
                  "% de los inyectados": es_inyectado[orden[:k]].sum() / n_inyectados * 100})
pd.DataFrame(filas).set_index("se revisa el").round(1)

**Cómo leerlo.** Revisar el 1 % (≈ 230 incidentes) ya encuentra la mayoría de las anomalías inyectadas; pasar de ahí rinde cada vez menos. **El corte lo decide el negocio:** cuántos incidentes puede revisar el equipo y cuánto cuesta dejar uno raro sin mirar. Si el equipo revisa 20 por día, se mira el 0,1 % mejor puntuado, no el 5 %.

## 7 · Raro «para su grupo»
Un incidente de **rutina** que tarda tres días es raro. Un incidente **complejo** que tarda tres días es normal. **Lo que es raro depende del contexto.** Con los grupos de la Práctica 3 podemos puntuar cada incidente **frente a los de su propio grupo** (anomalía **contextual**):

In [ ]:
from sklearn.cluster import KMeans

grupos = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(Z)                  # los mismos 5 grupos de la Práctica 3
orden_grupos = pd.Series(tabla.dias_a_resolver.values).groupby(grupos).median().sort_values().index
grupo = pd.Series(grupos).map({g: i for i, g in enumerate(orden_grupos)}).values
NOMBRES = {0: "Rutina rápida", 1: "Prioridad alta", 2: "Cambian de manos, sin esperas", 3: "Con esperas", 4: "Reabiertos y complejos"}

# Puntaje de cada incidente frente a SU grupo: un Isolation Forest por grupo, convertido a «percentil dentro del grupo»
percentil_grupo = np.zeros(len(tabla))
for g in range(5):
    en_grupo = grupo == g
    p = -IsolationForest(n_estimators=100, random_state=0).fit(Z[en_grupo]).score_samples(Z[en_grupo])
    percentil_grupo[en_grupo] = pd.Series(p).rank(pct=True).values * 100
percentil_general = tabla.puntaje_isolation.rank(pct=True).values * 100

tabla["grupo"] = [NOMBRES[g] for g in grupo]
tabla["percentil_general"], tabla["percentil_en_su_grupo"] = percentil_general, percentil_grupo
# Raros SOLO para su grupo: nada especial en general, pero entre los más raros de su grupo
contextuales = tabla[(tabla.percentil_general < 95) & (tabla.percentil_en_su_grupo > 98)].sort_values("percentil_en_su_grupo", ascending=False)
print(f"{len(contextuales)} incidentes que pasarían desapercibidos en general (no están entre el 5 % más raro) pero sí entre el 2 % más raro de su grupo. Algunos:")
contextuales[["grupo", "cambios_de_estado", "reasignaciones", "esperas", "dias_a_resolver", "percentil_general", "percentil_en_su_grupo"]].head(8).round(1)

**Qué leer ahí.** Son incidentes que, mirados contra **todos**, parecen normales, pero **frente a los de su propio grupo** destacan. En la tabla, casi todos son de «rutina rápida» y **tardaron entre 4 y 12 días y tuvieron una espera**: para un incidente complejo eso sería normal; para uno de rutina, no. Es la idea central de la **analítica de comportamiento** (UEBA): no se compara a una persona con *todos*, se la compara con **quienes se le parecen**.

## Cierre
- **Anomalía** = lo que no se parece a lo habitual. **Anómalo no es malicioso**: es dónde mirar.
- Lo «normal» **no suele ser una campana**; una regla de una variable se queda corta.
- Hay métodos que miran **todas las variables a la vez**: *Isolation Forest* (aislar) y el autoencoder (reconstruir). **Cada uno define «raro» a su manera.**
- **Evaluar sin etiquetas** exige ingenio: anomalías inyectadas, indicadores indirectos y, sobre todo, **revisión humana**. Cada una **mide algo distinto**.
- El **corte** (cuántos revisar) es una decisión de negocio, no del modelo.
- Lo raro depende del **contexto**: se compara con los incidentes parecidos.
- En ciberseguridad esto es la base de la detección de comportamientos anómalos de usuarios y sistemas.